In [9]:
# ======================================================================================
# CÓDIGO DE MONITOREO INTEGRADO DE ENERGÍA CON SENSORES TUYA
# ======================================================================================
# -*- coding: utf-8 -*-

"""
Monitoreo integrado de energía con sensores Tuya.

OBJETIVO GENERAL
----------------
Este script consulta periódicamente la API de Tuya para leer datos de distintos medidores
instalados en varias viviendas. Después de obtener los datos, guarda dos versiones:

1. data_raw/<Vivienda>/<Sensor>.csv
   - Contiene la respuesta "cruda" de Tuya (sin conversiones de unidades).

2. data_conv/<Vivienda>/<Sensor>.csv
   - Contiene los datos transformados a unidades físicas interpretables
     (por ejemplo: voltaje en V, corriente en A, frecuencia en Hz, etc.).

Además, el script puede generar gráficas en vivo a partir de los archivos convertidos.

NUEVA FUNCIONALIDAD INCORPORADA
-------------------------------
Además del guardado individual por sensor, el script puede construir archivos unificados
por vivienda cuando existan dos fases independientes (por ejemplo V2FaseA y V2FaseB),
dejando ambas lecturas en una sola fila por timestamp.

También consolida automáticamente los archivos unificados y genera:
- una tabla global CSV
- un archivo GeoJSON global

CARACTERÍSTICAS PRINCIPALES
---------------------------
- Organización por vivienda y por sensor.
- Una única conexión OpenAPI a Tuya.
- Manejo robusto de errores transitorios.
- Manejo de expiración o invalidación de token/sesión.
- Reintentos automáticos ante fallos de red o autenticación.
- Soporte para tres tipos de medidores:
    * M1 : medidor trifásico con variables codificadas en base64.
    * V2 : medidores con una estructura/escala específica.
    * V3 : medidores con otra estructura/escala específica.
- Creación automática de archivos unificados por vivienda.
- Exportación automática a GeoJSON.
- Los sensores cuyo device_id aún no se conoce se omiten sin detener el programa.

NOTA IMPORTANTE
---------------
Este script está pensado para ejecutarse en Colab o Jupyter, por eso incluye la línea
de instalación con "!pip". Si lo vas a correr como script .py fuera de notebook, esa línea
debe reemplazarse por una instalación previa en terminal.

AUTOR / CONTEXTO
----------------
Código preparado para monitoreo residencial de variables eléctricas con dispositivos Tuya
y posterior análisis de consumo energético.
"""

# ======================================================================================
# 1) INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ======================================================================================

!pip install -q tuya-connector-python

import tuya_connector  # noqa: F401

import os
import csv
import time
import base64
import json
from datetime import datetime
from zoneinfo import ZoneInfo

from tuya_connector import TuyaOpenAPI

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ======================================================================================
# 2) PARÁMETROS DE VISUALIZACIÓN EN VIVO
# ======================================================================================

GRAFICAR_CADA_N_RONDAS = 5
_ronda = 0


def graficar_estado_actual(data_folder_conv, viviendas):
    """
    Lee los archivos CSV convertidos (data_conv) y genera gráficas en vivo.

    Observación importante
    ----------------------
    Esta función grafica solamente los archivos individuales de sensor.
    Los archivos unificados se excluyen para evitar problemas de lectura,
    ya que usan separador ';' y formato de exportación distinto.
    """
    clear_output(wait=True)
    print("📈 Gráficas (actualización en vivo) — leyendo CSV CONVERTIDOS...\n")

    archivos_unificados = {cfg["archivo_salida"] for cfg in UNIFICACIONES.values()}

    for viv in viviendas.keys():
        viv_folder = os.path.join(data_folder_conv, viv)

        if not os.path.isdir(viv_folder):
            continue

        archivos = [
            f for f in os.listdir(viv_folder)
            if f.endswith(".csv")
            and f not in archivos_unificados
            and not f.endswith("_tabla.csv")
        ]

        if not archivos:
            continue

        print(f"🏠 {viv}")

        for archivo in sorted(archivos):
            path = os.path.join(viv_folder, archivo)

            try:
                df = pd.read_csv(path)

                if df.empty:
                    continue

                if "timestamp" in df.columns:
                    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

                cols_num = [c for c in df.columns if c != "timestamp"]

                for c in cols_num:
                    df[c] = pd.to_numeric(df[c], errors="coerce")

                dfp = df.tail(200).copy()

                if "timestamp" not in dfp.columns or dfp["timestamp"].isna().all():
                    continue

                plt.figure(figsize=(10, 4))

                for c in cols_num:
                    plt.plot(dfp["timestamp"], dfp[c], label=c)

                plt.title(f"{archivo.replace('.csv','')}")
                plt.xlabel("timestamp")
                plt.xticks(rotation=30)
                plt.legend(fontsize=8)
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  ⚠️ Error graficando {archivo}: {e}")

        print()


# ======================================================================================
# 3) CONFIGURACIÓN GENERAL DEL PROGRAMA
# ======================================================================================

API_ENDPOINT = "https://openapi.tuyaus.com"

# -----------------------------------------------------------------------------
# CREDENCIALES
# -----------------------------------------------------------------------------
os.environ["TUYA_ACCESS_ID"] = "devg78x5vga3syrxjdsg"
os.environ["TUYA_ACCESS_KEY"] = "723b31007c6c48c7bbac834688130ca7"

ACCESS_ID = os.getenv("TUYA_ACCESS_ID")
ACCESS_KEY = os.getenv("TUYA_ACCESS_KEY")

# -----------------------------------------------------------------------------
# PARÁMETROS DE REINTENTO Y TEMPORIZACIÓN
# -----------------------------------------------------------------------------
INTERVALO_MINUTOS = 1
MAX_REINTENTOS = 3
SLEEP_REINTENTO_S = 2

# -----------------------------------------------------------------------------
# PARÁMETROS DE RECONEXIÓN DE SESIÓN/TOKEN
# -----------------------------------------------------------------------------
MAX_RECONNECTS = 2
SLEEP_RECONNECT_S = 1.0
TZ_LOCAL = ZoneInfo("America/Monterrey")

# -----------------------------------------------------------------------------
# CARPETAS DE SALIDA
# -----------------------------------------------------------------------------
from google.colab import drive

MOUNT_POINT = "/content/gdrive"

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

BASE_DIR = f"{MOUNT_POINT}/MyDrive/Monitoreo_Tuya"
DATA_RAW_FOLDER = os.path.join(BASE_DIR, "data_raw")
DATA_CONV_FOLDER = os.path.join(BASE_DIR, "data_conv")

# ======================================================================================
# 4) CONFIGURACIÓN DE VIVIENDAS Y SENSORES
# ======================================================================================

VIVIENDAS = {
    "Vivienda1 - Doctor": {
        #"Medidor1": {"tipo": "M1", "device_id": "eb386433cd7a23bf51n4hf"},
        "V1Extra": {"tipo": "V1EXTRA_DUAL", "device_id": "eb47caff2a29e9fe2aos9i"},
    },
    "Vivienda2 - Ángel": {
        "V2FaseA": {"tipo": "V2", "device_id": "eb1410cb74c0a62739mty3"},
        "V2FaseB": {"tipo": "V2", "device_id": "eb109bcaa77dc6fe8baoxs"},
    },
    "Vivienda3 - María": {
        "V3FaseA": {"tipo": "V3", "device_id": "ebc36f971d1011543dfifw"},
        "V3FaseB": {"tipo": "V3", "device_id": "ebe1be117fd2724821law2"},
    },
    #"Prueba": {
    #    "V3FaseA": {"tipo": "V3", "device_id": "eb16c87a7aeab1ddebaqt9"},
    #}
}

# ======================================================================================
# 4.1) CONFIGURACIÓN DE ARCHIVOS UNIFICADOS POR VIVIENDA
# ======================================================================================

UNIFICACIONES = {
    "Vivienda1 - Doctor": {
        "archivo_salida": "Medidor1.csv",
        "fase_A": "V1ExtraA",
        "fase_B": "V1ExtraB",
        "latitud": 25.6189821,
        "longitud": -100.2950499,
    },
    "Vivienda2 - Ángel": {
        "archivo_salida": "Medidor2.csv",
        "fase_A": "V2FaseA",
        "fase_B": "V2FaseB",
        "latitud": 25.646641,
        "longitud": -100.288357,
    },
    "Vivienda3 - María": {
        "archivo_salida": "Medidor3.csv",
        "fase_A": "V3FaseA",
        "fase_B": "V3FaseB",
        "latitud": 25.648629,
        "longitud": -100.279387,
    },
}

# ======================================================================================
# 5) CONEXIÓN INICIAL A TUYA
# ======================================================================================

print("Iniciando conexión con Tuya...")

openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

print("✅ Conexión con Tuya OK\n")

# ======================================================================================
# 6) FUNCIONES UTILITARIAS GENERALES
# ======================================================================================

def now_str():
    """
    Devuelve el timestamp actual en la zona horaria local definida.
    """
    return datetime.now(TZ_LOCAL).strftime("%Y-%m-%d %H:%M:%S")


def ensure_folder(path):
    """
    Crea una carpeta si no existe.
    """
    os.makedirs(path, exist_ok=True)


def _looks_like_token_issue(resp=None, exc: Exception = None) -> bool:
    """
    Heurística para detectar problemas de token o sesión.
    """
    if exc is not None:
        s = str(exc).lower()
        keywords = [
            "token", "expire", "expired", "invalid",
            "unauthorized", "auth", "forbidden", "signature"
        ]
        return any(k in s for k in keywords)

    if isinstance(resp, dict):
        success = resp.get("success", True)

        if success is False:
            code = str(resp.get("code", "")).lower()
            msg = str(resp.get("msg", "")).lower()
            message = str(resp.get("message", "")).lower()
            combined = " ".join([code, msg, message])

            keywords = [
                "token", "expire", "expired", "invalid",
                "unauthorized", "auth", "forbidden", "signature"
            ]
            return any(k in combined for k in keywords)

    return False


def _reconnect_openapi():
    """
    Reconstruye la conexión con la API de Tuya.
    """
    global openapi

    try:
        print("🔄 Re-conectando a Tuya (refresh token/sesión)...")
        openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
        openapi.connect()
        time.sleep(SLEEP_RECONNECT_S)
        print("✅ Re-conexión OK")
        return True, None

    except Exception as e:
        print(f"❌ Falló reconnect(): {e}")
        return False, e


def openapi_get_safe(path: str):
    """
    Ejecuta openapi.get(path) de forma segura, intentando resolver problemas
    de token/sesión mediante reconexión automática.
    """
    try:
        resp = openapi.get(path)

        if _looks_like_token_issue(resp=resp):
            for _ in range(MAX_RECONNECTS):
                ok, _e = _reconnect_openapi()
                if not ok:
                    break

                resp2 = openapi.get(path)
                if not _looks_like_token_issue(resp=resp2):
                    return resp2, None

            return resp, Exception(f"Posible problema de token/sesión: {resp}")

        return resp, None

    except Exception as e:
        if _looks_like_token_issue(exc=e):
            for _ in range(MAX_RECONNECTS):
                ok, _e = _reconnect_openapi()
                if not ok:
                    break

                try:
                    resp2 = openapi.get(path)
                    if not _looks_like_token_issue(resp=resp2):
                        return resp2, None
                except Exception as e2:
                    e = e2

            return None, e

        return None, e


def tuya_get_with_retry(path: str, max_retries=MAX_REINTENTOS):
    """
    Ejecuta una consulta GET a Tuya con reintentos normales adicionales.
    """
    last_err = None

    for _ in range(1, max_retries + 1):
        resp, err = openapi_get_safe(path)

        if err is None and resp is not None:
            return resp, None

        last_err = err
        time.sleep(SLEEP_REINTENTO_S)

    return None, last_err


def obtener_datos_V1EXTRA_DUAL(device_id, ts):
    """
    Obtiene datos de un dispositivo V1Extra que reporta dos canales.
    """
    path = f"/v2.0/cloud/thing/{device_id}/shadow/properties"
    resp, err = tuya_get_with_retry(path)

    if err or not resp:
        return None, None, err

    result = resp.get("result", {})
    props = result.get("properties", [])
    propiedades = {p["code"]: p.get("value") for p in props}

    datos_raw = {"timestamp": ts}
    for k, v in propiedades.items():
        datos_raw[k] = v

    voltage = propiedades.get("voltage_a", 0) / 10
    freq = propiedades.get("freq", 0) / 100

    datos_conv_split = {
        "V1ExtraA": {
            "timestamp": ts,
            "voltage_V": voltage,
            "current_A": propiedades.get("current_a", 0) / 1000,
            "power_W": propiedades.get("power_a", 0) / 10,
            "pf": propiedades.get("power_factor", 0) / 100,
            "freq_Hz": freq,
            "energy_Wh": propiedades.get("energy_forword_a", 0),
            "energy_reverse_Wh": propiedades.get("energy_reverse_a", 0),
            "direction": propiedades.get("direction_a", ""),
        },
        "V1ExtraB": {
            "timestamp": ts,
            "voltage_V": voltage,
            "current_A": propiedades.get("current_b", 0) / 1000,
            "power_W": propiedades.get("power_b", 0) / 10,
            "pf": propiedades.get("power_factor_b", 0) / 100,
            "freq_Hz": freq,
            "energy_Wh": propiedades.get("energy_forword_b", 0),
            "energy_reverse_Wh": propiedades.get("energy_reserse_b", 0),
            "direction": propiedades.get("direction_b", ""),
        },
    }

    return datos_raw, datos_conv_split, None


# ======================================================================================
# 7) FUNCIONES DEL MEDIDOR TIPO M1
# ======================================================================================

def decodificar_instantaneo(b64):
    """
    Decodifica una cadena base64 proveniente de los campos pa_instant o pb_instant.
    """
    try:
        if not b64:
            return (0.0, 0.0, 0.0)

        data = base64.b64decode(b64)

        if len(data) < 12:
            return (0.0, 0.0, 0.0)

        voltaje = int.from_bytes(data[0:2], "big") / 10.0
        corriente = int.from_bytes(data[2:5], "big") / 1000.0
        potencia = int.from_bytes(data[5:8], "big", signed=False) / 10.0

        return voltaje, corriente, potencia

    except Exception as e:
        print(f"⚠️ Error al decodificar {b64}: {e}")
        return (0.0, 0.0, 0.0)


# ======================================================================================
# 8) FUNCIONES DE OBTENCIÓN DE DATOS POR TIPO DE SENSOR
# ======================================================================================

def obtener_datos_M1(device_id, ts):
    """
    Obtiene datos del medidor tipo M1.
    """
    path = f"/v2.0/cloud/thing/{device_id}/shadow/properties"
    resp, err = tuya_get_with_retry(path)

    if err or not resp:
        return None, None, err

    result = resp.get("result", {})
    props_list = result.get("properties", [])
    props = {p["code"]: p.get("value") for p in props_list}

    datos_raw = {"timestamp": ts}
    for k, v in props.items():
        datos_raw[k] = v

    total_forward = props.get("total_forward_energy", 0)
    total_reverse = props.get("reverse_energy_total", 0)

    pa_raw = props.get("pa_instant", "")
    pb_raw = props.get("pb_instant", "")

    volt_a, corr_a, pot_a = decodificar_instantaneo(pa_raw)
    volt_b, corr_b, pot_b = decodificar_instantaneo(pb_raw)

    datos_conv = {
        "timestamp": ts,
        "voltaje_A(V)": volt_a,
        "corriente_A(A)": corr_a,
        "potencia_A(W)": pot_a,
        "voltaje_B(V)": volt_b,
        "corriente_B(A)": corr_b,
        "potencia_B(W)": pot_b,
        "energia_directa(Wh)": total_forward,
        "energia_inversa(Wh)": total_reverse,
    }

    return datos_raw, datos_conv, None


def obtener_datos_V2(device_id, ts):
    """
    Obtiene datos de sensores con transformación tipo V2.
    """
    path = f"/v2.0/cloud/thing/{device_id}/shadow/properties"
    resp, err = tuya_get_with_retry(path)

    if err or not resp:
        return None, None, err

    result = resp.get("result", {})
    props = result.get("properties", [])
    propiedades = {p["code"]: p.get("value") for p in props}

    datos_raw = {"timestamp": ts}
    for k, v in propiedades.items():
        datos_raw[k] = v

    datos_conv = {
        "timestamp": ts,
        "voltage_V": propiedades.get("voltage_a", 0) / 10,
        "current_A": propiedades.get("current_b", 0) / 1000,
        "power_W": propiedades.get("power_b", 0) / 10,
        "pf": propiedades.get("power_factor_b", 0) / 100,
        "freq_Hz": propiedades.get("freq", 0) / 100,
        "energy_Wh": propiedades.get("energy_forword_b", 0),
    }

    return datos_raw, datos_conv, None


def obtener_datos_V3(device_id, ts):
    """
    Obtiene datos de sensores con transformación tipo V3.
    """
    path = f"/v2.0/cloud/thing/{device_id}/shadow/properties"
    resp, err = tuya_get_with_retry(path)

    if err or not resp:
        return None, None, err

    result = resp.get("result", {})
    props = result.get("properties", [])
    propiedades = {p["code"]: p.get("value") for p in props}

    datos_raw = {"timestamp": ts}
    for k, v in propiedades.items():
        datos_raw[k] = v

    datos_conv = {
        "timestamp": ts,
        "voltage_V": propiedades.get("voltage_a", 0) / 10,
        "current_A": propiedades.get("current_a", 0) / 100,
        "power_W": propiedades.get("power_a", 0),
        "pf": propiedades.get("power_factor", 0) / 100,
        "freq_Hz": propiedades.get("freq", 0) / 100,
        "energy_Wh": propiedades.get("energy_forword_a", 0),
    }

    return datos_raw, datos_conv, None


def obtener_datos_V1EXTRA(device_id, ts):
    """
    Obtiene datos de sensores tipo V1Extra.
    """
    path = f"/v2.0/cloud/thing/{device_id}/shadow/properties"
    resp, err = tuya_get_with_retry(path)

    if err or not resp:
        return None, None, err

    result = resp.get("result", {})
    props = result.get("properties", [])
    propiedades = {p["code"]: p.get("value") for p in props}

    datos_raw = {"timestamp": ts}
    for k, v in propiedades.items():
        datos_raw[k] = v

    datos_conv = {
        "timestamp": ts,
        "voltage_V": propiedades.get("voltage_a", 0) / 10,
        "current_A": propiedades.get("current_b", 0) / 1000,
        "power_W": propiedades.get("power_b", 0) / 10,
        "pf": propiedades.get("power_factor_b", 0) / 100,
        "freq_Hz": propiedades.get("freq", 0) / 100,
        "energy_Wh": propiedades.get("energy_forword_b", 0),
    }

    return datos_raw, datos_conv, None


# ======================================================================================
# 9) FUNCIÓN DE GUARDADO EN CSV
# ======================================================================================

def guardar_csv(base_folder, vivienda, sensor_name, datos: dict):
    """
    Guarda un diccionario de datos en un CSV individual por sensor.
    """
    carpeta = os.path.join(base_folder, vivienda)
    ensure_folder(carpeta)

    archivo = os.path.join(carpeta, f"{sensor_name}.csv")
    nuevo = not os.path.exists(archivo)
    cols = list(datos.keys())

    with open(archivo, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cols)

        if nuevo:
            writer.writeheader()

        writer.writerow(datos)


# ======================================================================================
# 9.1) FUNCIONES PARA ARCHIVOS UNIFICADOS POR VIVIENDA
# ======================================================================================

def formatear_valor_csv_es(valor):
    """
    Formatea valores para exportarlos con estilo similar al Medidor2.csv:
    - separador de columnas ';'
    - decimales con coma
    """
    if valor is None:
        return ""

    if isinstance(valor, str):
        return valor

    if isinstance(valor, bool):
        return str(valor)

    if isinstance(valor, int):
        return str(valor)

    if isinstance(valor, float):
        if pd.isna(valor):
            return ""
        texto = f"{valor:.6f}".rstrip("0").rstrip(".")
        return texto.replace(".", ",")

    return str(valor)


def construir_fila_unificada(ts, latitud, longitud, datos_fase_a, datos_fase_b):
    dt = datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")

    fila = {
        "latitud": latitud,
        "longitud": longitud,
        "timestamp": f"{dt.day}/{dt.month}/{dt.year} {dt.hour}:{dt.minute:02d}",
        "day": dt.day,
        "month": dt.month,
        "year": dt.year,
        "hour": dt.hour,
        "minute": dt.minute,
    }

    # Detectar tipo automáticamente
    if "direction" in datos_fase_a:
        # CASO V1EXTRA
        fila.update({
            "voltage_V_A": datos_fase_a.get("voltage_V"),
            "current_A_A": datos_fase_a.get("current_A"),
            "power_W_A": datos_fase_a.get("power_W"),
            "pf_A": datos_fase_a.get("pf"),
            "freq_Hz_A": datos_fase_a.get("freq_Hz"),
            "energy_Wh_A": datos_fase_a.get("energy_Wh"),
            "energy_reverse_Wh_A": datos_fase_a.get("energy_reverse_Wh"),
            "direction_A": datos_fase_a.get("direction"),

            "voltage_V_B": datos_fase_b.get("voltage_V"),
            "current_A_B": datos_fase_b.get("current_A"),
            "power_W_B": datos_fase_b.get("power_W"),
            "pf_B": datos_fase_b.get("pf"),
            "freq_Hz_B": datos_fase_b.get("freq_Hz"),
            "energy_Wh_B": datos_fase_b.get("energy_Wh"),
            "energy_reverse_Wh_B": datos_fase_b.get("energy_reverse_Wh"),
            "direction_B": datos_fase_b.get("direction"),
        })

    else:
        # CASO V2 / V3
        fila.update({
            "voltage_V_A": datos_fase_a.get("voltage_V"),
            "current_A_A": datos_fase_a.get("current_A"),
            "power_W_A": datos_fase_a.get("power_W"),
            "pf_A": datos_fase_a.get("pf"),
            "freq_Hz_A": datos_fase_a.get("freq_Hz"),
            "energy_Wh_A": datos_fase_a.get("energy_Wh"),

            "voltage_V_B": datos_fase_b.get("voltage_V"),
            "current_A_B": datos_fase_b.get("current_A"),
            "power_W_B": datos_fase_b.get("power_W"),
            "pf_B": datos_fase_b.get("pf"),
            "freq_Hz_B": datos_fase_b.get("freq_Hz"),
            "energy_Wh_B": datos_fase_b.get("energy_Wh"),
        })

    return fila


def guardar_csv_unificado(base_folder, vivienda, archivo_salida, fila: dict):
    """
    Guarda una fila en un CSV unificado por vivienda.
    Si el archivo ya existe pero no tiene las columnas correctas,
    lo reconstruye automáticamente.
    """
    carpeta = os.path.join(base_folder, vivienda)
    ensure_folder(carpeta)

    archivo = os.path.join(carpeta, archivo_salida)
    cols = list(fila.keys())

    fila_formateada = {k: formatear_valor_csv_es(v) for k, v in fila.items()}

    # ---------------------------------------------------------
    # VALIDAR SI EL ARCHIVO EXISTE Y TIENE LAS COLUMNAS CORRECTAS
    # ---------------------------------------------------------
    if os.path.exists(archivo):
        try:
            df_existente = pd.read_csv(archivo, sep=";")

            cols_existentes = set(df_existente.columns)
            cols_nuevas = set(cols)

            # Si faltan columnas clave → RECONSTRUIR ARCHIVO
            if not cols_nuevas.issubset(cols_existentes):
                print(f"♻️ Reescribiendo {archivo_salida} (estructura antigua detectada)")

                with open(archivo, "w", newline="", encoding="utf-8-sig") as f:
                    writer = csv.DictWriter(f, fieldnames=cols, delimiter=";")
                    writer.writeheader()
                    writer.writerow(fila_formateada)

                return

        except Exception:
            print(f"⚠️ Error leyendo {archivo_salida}, se reconstruye.")

            with open(archivo, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=cols, delimiter=";")
                writer.writeheader()
                writer.writerow(fila_formateada)

            return

    # ---------------------------------------------------------
    # CASO NORMAL: APPEND
    # ---------------------------------------------------------
    nuevo = not os.path.exists(archivo)

    with open(archivo, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=cols, delimiter=";")

        if nuevo:
            writer.writeheader()

        writer.writerow(fila_formateada)


# ======================================================================================
# 10) RUTEADOR DE FUNCIONES SEGÚN TIPO DE SENSOR
# ======================================================================================

GETTERS = {
    "M1": obtener_datos_M1,
    "V2": obtener_datos_V2,
    "V3": obtener_datos_V3,
    "V1EXTRA_DUAL": obtener_datos_V1EXTRA_DUAL,
}

# ======================================================================================
# 10.1) FUNCIONES PARA CONSOLIDAR ARCHIVOS UNIFICADOS Y EXPORTAR A GEOJSON
# ======================================================================================

def leer_csv_unificado(path_csv):
    """
    Lee un CSV unificado exportado con separador ';' y convierte columnas numéricas.
    """
    if not os.path.exists(path_csv):
        return None

    try:
        df = pd.read_csv(path_csv, sep=";", on_bad_lines="skip")

        if df.empty:
            return None

        # Limpiar coordenadas
        for col in ["latitud", "longitud"]:
            if col in df.columns:
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.replace(",", ".", regex=False)
                    .replace("nan", None)
                )
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Limpiar columnas numéricas potenciales
        cols_posibles = [
            "day", "month", "year", "hour", "minute",
            "voltage_V_A", "current_A_A", "power_W_A", "pf_A", "freq_Hz_A", "energy_Wh_A",
            "voltage_V_B", "current_A_B", "power_W_B", "pf_B", "freq_Hz_B", "energy_Wh_B",
            "energy_reverse_Wh_A", "energy_reverse_Wh_B"
        ]

        for col in cols_posibles:
            if col in df.columns:
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.replace(",", ".", regex=False)
                    .replace("nan", None)
                )
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Timestamp
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], dayfirst=True, errors="coerce")

        return df

    except Exception as e:
        print(f"⚠️ Error leyendo CSV unificado {path_csv}: {e}")
        return None


def construir_feature_geojson(group, lat, lon, id_punto):
    """
    Construye una feature GeoJSON para un punto agrupado por coordenadas.

    En lugar de promedios y máximos, toma los valores de la fila más reciente.
    Además calcula la energía acumulada entre las dos últimas observaciones válidas
    y la suma de deltas entre fases.
    """
    # Ordenar por tiempo por seguridad
    if "timestamp" in group.columns:
        group = group.sort_values("timestamp").copy()
    else:
        group = group.copy()

    def ultimo_valor(col):
        if col in group.columns and group[col].notna().any():
            return a_python_scalar(group[col].dropna().iloc[-1])
        return None

    def delta_entre_ultimas(col):
        """
        Calcula diferencia entre las dos últimas observaciones válidas.
        Si el delta resulta negativo, retorna None.
        """
        if col in group.columns:
            serie = group[col].dropna()
            if len(serie) >= 2:
                delta = serie.iloc[-1] - serie.iloc[-2]
                delta = a_python_scalar(delta)
                return delta if delta is not None and delta >= 0 else None
        return None

    def sumar_deltas(*args):
        """
        Suma únicamente los deltas disponibles (no None).
        Si ninguno existe, retorna None.
        """
        valores = [x for x in args if x is not None]
        if len(valores) == 0:
            return None
        return sum(valores)

    timestamp_penultimo = None
    if "timestamp" in group.columns:
        serie_ts = group["timestamp"].dropna()
        if len(serie_ts) >= 2:
            timestamp_penultimo = str(serie_ts.iloc[-2])

    # ---------------------------------------------------------
    # DELTAS INDIVIDUALES
    # ---------------------------------------------------------
    delta_energy_a = delta_entre_ultimas("energy_Wh_A")
    delta_energy_b = delta_entre_ultimas("energy_Wh_B")
    delta_reverse_a = delta_entre_ultimas("energy_reverse_Wh_A")
    delta_reverse_b = delta_entre_ultimas("energy_reverse_Wh_B")

    # ---------------------------------------------------------
    # SUMAS DE DELTAS
    # ---------------------------------------------------------
    delta_energy_total = sumar_deltas(delta_energy_a, delta_energy_b)
    delta_reverse_total = sumar_deltas(delta_reverse_a, delta_reverse_b)

    props = {
        "id_punto": str(id_punto),
        "n_registros": int(len(group)),

        # Últimos valores medidos
        "voltage_V_A": ultimo_valor("voltage_V_A"),
        "current_A_A": ultimo_valor("current_A_A"),
        "power_W_A": ultimo_valor("power_W_A"),
        "pf_A": ultimo_valor("pf_A"),
        "freq_Hz_A": ultimo_valor("freq_Hz_A"),
        "energy_Wh_A": ultimo_valor("energy_Wh_A"),

        "voltage_V_B": ultimo_valor("voltage_V_B"),
        "current_A_B": ultimo_valor("current_A_B"),
        "power_W_B": ultimo_valor("power_W_B"),
        "pf_B": ultimo_valor("pf_B"),
        "freq_Hz_B": ultimo_valor("freq_Hz_B"),
        "energy_Wh_B": ultimo_valor("energy_Wh_B"),

        "energy_reverse_Wh_A": ultimo_valor("energy_reverse_Wh_A"),
        "energy_reverse_Wh_B": ultimo_valor("energy_reverse_Wh_B"),

        # Deltas individuales
        "energy_Wh_A_delta_ultimas_2": delta_energy_a,
        "energy_Wh_B_delta_ultimas_2": delta_energy_b,
        "energy_reverse_Wh_A_delta_ultimas_2": delta_reverse_a,
        "energy_reverse_Wh_B_delta_ultimas_2": delta_reverse_b,

        # NUEVO: sumas de deltas
        "energy_Wh_delta_total_ultimas_2": delta_energy_total,
        "energy_reverse_Wh_delta_total_ultimas_2": delta_reverse_total,

        # Tiempo
        "timestamp_ultimo": str(ultimo_valor("timestamp")) if "timestamp" in group.columns else None,
        "timestamp_penultimo": timestamp_penultimo,
    }

    # Direcciones, si existen
    if "direction_A" in group.columns:
        props["direction_A"] = ultimo_valor("direction_A")

    if "direction_B" in group.columns:
        props["direction_B"] = ultimo_valor("direction_B")

    props = serializar_props(props)

    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(lon), float(lat)]
        },
        "properties": props
    }

def a_python_scalar(valor):
    """
    Convierte escalares de pandas/numpy a tipos nativos de Python
    para que json.dump los pueda serializar.
    """
    if pd.isna(valor):
        return None

    # Tipos numpy/pandas con método item()
    if hasattr(valor, "item"):
        try:
            return valor.item()
        except Exception:
            pass

    # Timestamp u otros objetos de fecha
    if isinstance(valor, (pd.Timestamp, datetime)):
        return str(valor)

    return valor


def serializar_props(props: dict):
    """
    Convierte todos los valores de un diccionario a tipos serializables por JSON.
    """
    return {k: a_python_scalar(v) for k, v in props.items()}

def exportar_csv_unificado_a_geojson(path_csv, path_csv_salida, path_geojson_salida):
    """
    Convierte un único CSV unificado a:
    - una tabla CSV procesada
    - un archivo GeoJSON

    No concatena viviendas. Trabaja archivo por archivo.
    """
    df = leer_csv_unificado(path_csv)

    if df is None or df.empty:
        print(f"⚠️ No se pudo exportar {path_csv}: archivo vacío o no disponible.")
        return

    # Limpiar filas sin coordenadas
    df = df.dropna(subset=["latitud", "longitud"]).copy()

    if df.empty:
        print(f"⚠️ No se pudo exportar {path_csv}: no hay coordenadas válidas.")
        return

    # ID por punto
    df["id_punto"] = (
        df["latitud"].astype(str) + "_" + df["longitud"].astype(str)
    )

    features = []
    grouped = df.groupby(["id_punto", "latitud", "longitud"], dropna=True)

    for (id_p, lat, lon), group in grouped:
        feature = construir_feature_geojson(group, lat, lon, id_p)
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    # Guardar tabla procesada
    df.to_csv(path_csv_salida, index=False, encoding="utf-8-sig")

    # Guardar geojson
    with open(path_geojson_salida, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)

    print(f"🗺️ Tabla procesada guardada en: {path_csv_salida}")
    print(f"🗺️ GeoJSON guardado en: {path_geojson_salida}")


def exportar_geojson_por_medidor(data_folder_conv, unificaciones):
    """
    Recorre cada archivo unificado configurado en UNIFICACIONES
    y genera su propio CSV procesado y su propio GeoJSON.
    """
    for viv, cfg in unificaciones.items():
        archivo_csv = os.path.join(data_folder_conv, viv, cfg["archivo_salida"])

        nombre_base = os.path.splitext(cfg["archivo_salida"])[0]
        archivo_csv_salida = os.path.join(data_folder_conv, viv, f"{nombre_base}_tabla.csv")
        archivo_geojson_salida = os.path.join(data_folder_conv, viv, f"{nombre_base}.geojson")

        exportar_csv_unificado_a_geojson(
            path_csv=archivo_csv,
            path_csv_salida=archivo_csv_salida,
            path_geojson_salida=archivo_geojson_salida
        )

def exportar_global_unificado(data_folder_conv, unificaciones):
    """
    Une todos los CSV unificados por vivienda en un solo CSV global
    y genera un único GeoJSON global.
    """
    dfs = []

    for viv, cfg in unificaciones.items():
        archivo_csv = os.path.join(data_folder_conv, viv, cfg["archivo_salida"])
        df = leer_csv_unificado(archivo_csv)

        if df is None or df.empty:
            print(f"⚠️ No se pudo incluir {archivo_csv} en el global.")
            continue

        df["vivienda"] = viv
        df["archivo_origen"] = cfg["archivo_salida"]
        dfs.append(df)

    if not dfs:
        print("⚠️ No hay datos para construir el CSV/GeoJSON global.")
        return

    # ---------------------------------------------------------
    # CSV GLOBAL: unir todas las filas de todos los medidores
    # ---------------------------------------------------------
    df_global = pd.concat(dfs, ignore_index=True)

    csv_global_out = os.path.join(data_folder_conv, "medidores_global.csv")
    df_global.to_csv(csv_global_out, index=False, encoding="utf-8-sig")
    print(f"📄 CSV global guardado en: {csv_global_out}")

    # ---------------------------------------------------------
    # GEOJSON GLOBAL: una feature por vivienda / punto
    # usando la lógica de última observación
    # ---------------------------------------------------------
    df_geo = df_global.dropna(subset=["latitud", "longitud"]).copy()

    if df_geo.empty:
        print("⚠️ No se pudo construir el GeoJSON global: no hay coordenadas válidas.")
        return

    df_geo["id_punto"] = (
        df_geo["latitud"].astype(str) + "_" + df_geo["longitud"].astype(str)
    )

    features = []
    grouped = df_geo.groupby(["id_punto", "latitud", "longitud"], dropna=True)

    for (id_p, lat, lon), group in grouped:
        feature = construir_feature_geojson(group, lat, lon, id_p)

        # Agregar vivienda más reciente al properties del feature
        if "vivienda" in group.columns and group["vivienda"].notna().any():
            vivienda_ult = group.sort_values("timestamp")["vivienda"].dropna().iloc[-1]
            feature["properties"]["vivienda"] = str(vivienda_ult)

        if "archivo_origen" in group.columns and group["archivo_origen"].notna().any():
            archivo_ult = group.sort_values("timestamp")["archivo_origen"].dropna().iloc[-1]
            feature["properties"]["archivo_origen"] = str(archivo_ult)

        features.append(feature)

    geojson_global = {
        "type": "FeatureCollection",
        "features": features
    }

    geojson_global_out = os.path.join(data_folder_conv, "medidores_global.geojson")
    with open(geojson_global_out, "w", encoding="utf-8") as f:
        json.dump(geojson_global, f, ensure_ascii=False, indent=2)

    print(f"🗺️ GeoJSON global guardado en: {geojson_global_out}")

# ======================================================================================
# 11) LOOP PRINCIPAL DE MONITOREO
# ======================================================================================

print(f"⚡ Iniciando monitoreo cada {INTERVALO_MINUTOS} minuto(s)...")
print("🕒 Los timestamps se están generando en hora local de America/Monterrey.")
print("💾 Guardado en dos salidas: data_raw (sin conversiones) y data_conv (con conversiones).")
print("🧩 Además, se generarán archivos unificados por vivienda cuando aplique.")
print("🗺️ También se generará automáticamente una tabla global y un GeoJSON global.")

try:
    while True:
        ts = now_str()

        print(f"\n================== RONDA {ts} ==================")

        for viv, sensores in VIVIENDAS.items():
            print(f"\n🏠 {viv}")

            # Acá se almacenan temporalmente las fases leídas en esta ronda
            datos_unificacion_ronda = {}

            for sensor_name, info in sensores.items():
                tipo = info.get("tipo")
                device_id = info.get("device_id")

                if device_id is None or str(device_id).strip() == "":
                    print(f"⏳ {sensor_name} ({tipo}): esperando device_id...")
                    continue

                getter = GETTERS.get(tipo)
                if getter is None:
                    print(f"⚠️ {sensor_name}: tipo desconocido '{tipo}'")
                    continue

                datos_raw, datos_conv, err = getter(device_id, ts)

                if err or datos_raw is None or datos_conv is None:
                    print(f"❌ {sensor_name} ({tipo}): error -> {err}")
                    continue

                if tipo == "V1EXTRA_DUAL":
                    guardar_csv(DATA_RAW_FOLDER, viv, sensor_name, datos_raw)

                    cfg_uni = UNIFICACIONES.get(viv)

                    for sub_sensor_name, sub_datos in datos_conv.items():
                        guardar_csv(DATA_CONV_FOLDER, viv, sub_sensor_name, sub_datos)

                        if cfg_uni is not None:
                            if sub_sensor_name == cfg_uni["fase_A"]:
                                datos_unificacion_ronda["A"] = sub_datos
                            elif sub_sensor_name == cfg_uni["fase_B"]:
                                datos_unificacion_ronda["B"] = sub_datos

                        print(
                            f"✅ {sub_sensor_name}: "
                            f"{float(sub_datos['voltage_V']):.1f}V | "
                            f"{float(sub_datos['current_A']):.3f}A | "
                            f"{float(sub_datos['power_W']):.1f}W | "
                            f"PF={float(sub_datos['pf']):.2f} | "
                            f"F={float(sub_datos['freq_Hz']):.2f}Hz | "
                            f"E={sub_datos['energy_Wh']} | "
                            f"Dir={sub_datos['direction']}"
                        )

                else:
                    guardar_csv(DATA_RAW_FOLDER, viv, sensor_name, datos_raw)
                    guardar_csv(DATA_CONV_FOLDER, viv, sensor_name, datos_conv)

                    cfg_uni = UNIFICACIONES.get(viv)
                    if cfg_uni is not None:
                        if sensor_name == cfg_uni["fase_A"]:
                            datos_unificacion_ronda["A"] = datos_conv
                        elif sensor_name == cfg_uni["fase_B"]:
                            datos_unificacion_ronda["B"] = datos_conv

                    if tipo == "M1":
                        print(
                            f"✅ {sensor_name}: "
                            f"A {float(datos_conv['voltaje_A(V)']):.1f}V "
                            f"{float(datos_conv['corriente_A(A)']):.3f}A "
                            f"{float(datos_conv['potencia_A(W)']):.1f}W | "
                            f"B {float(datos_conv['voltaje_B(V)']):.1f}V "
                            f"{float(datos_conv['corriente_B(A)']):.3f}A "
                            f"{float(datos_conv['potencia_B(W)']):.1f}W"
                        )
                    else:
                        print(
                            f"✅ {sensor_name}: "
                            f"{float(datos_conv['voltage_V']):.1f}V | "
                            f"{float(datos_conv['current_A']):.3f}A | "
                            f"{float(datos_conv['power_W']):.1f}W | "
                            f"PF={float(datos_conv['pf']):.2f} | "
                            f"F={float(datos_conv['freq_Hz']):.2f}Hz | "
                            f"E={datos_conv['energy_Wh']}"
                        )

            # ---------------------------------------------------------
            # CONSTRUCCIÓN DEL ARCHIVO UNIFICADO POR VIVIENDA
            # ---------------------------------------------------------
            cfg_uni = UNIFICACIONES.get(viv)
            if cfg_uni is not None:
                fase_a = datos_unificacion_ronda.get("A")
                fase_b = datos_unificacion_ronda.get("B")

                if fase_a is not None and fase_b is not None:
                    fila_unificada = construir_fila_unificada(
                        ts=ts,
                        latitud=cfg_uni["latitud"],
                        longitud=cfg_uni["longitud"],
                        datos_fase_a=fase_a,
                        datos_fase_b=fase_b,
                    )

                    guardar_csv_unificado(
                        DATA_CONV_FOLDER,
                        viv,
                        cfg_uni["archivo_salida"],
                        fila_unificada
                    )

                    print(f"🧩 Archivo unificado actualizado: {cfg_uni['archivo_salida']}")
                else:
                    print("⚠️ No se pudo construir archivo unificado en esta ronda: falta Fase A o Fase B.")

        # -----------------------------------------------------------------
        # ACTUALIZACIÓN DE GRÁFICAS
        # -----------------------------------------------------------------
        _ronda += 1

        if _ronda % GRAFICAR_CADA_N_RONDAS == 0:
            graficar_estado_actual(DATA_CONV_FOLDER, VIVIENDAS)

        # -----------------------------------------------------------------
        # EXPORTACIÓN A GEOJSON POR CADA MEDIDOR UNIFICADO
        # -----------------------------------------------------------------
        exportar_geojson_por_medidor(DATA_CONV_FOLDER, UNIFICACIONES)

        # -----------------------------------------------------------------
        # EXPORTACIÓN GLOBAL UNIFICADA
        # -----------------------------------------------------------------
        exportar_global_unificado(DATA_CONV_FOLDER, UNIFICACIONES)

        # -----------------------------------------------------------------
        # ESPERA HASTA LA SIGUIENTE RONDA
        # -----------------------------------------------------------------
        print(f"\n⌛ Esperando {INTERVALO_MINUTOS} minuto(s)...")
        time.sleep(INTERVALO_MINUTOS * 60)

except KeyboardInterrupt:
    print("\n🛑 Monitoreo detenido por el usuario.")

Iniciando conexión con Tuya...
✅ Conexión con Tuya OK

⚡ Iniciando monitoreo cada 1 minuto(s)...
🕒 Los timestamps se están generando en hora local de America/Monterrey.
💾 Guardado en dos salidas: data_raw (sin conversiones) y data_conv (con conversiones).
🧩 Además, se generarán archivos unificados por vivienda cuando aplique.
🗺️ También se generará automáticamente una tabla global y un GeoJSON global.

================== RONDA 2026-04-21 16:42:20 ==================

🏠 Vivienda1 - Doctor
✅ V1ExtraA: 133.7V | 6.826A | 853.8W | PF=0.93 | F=58.81Hz | E=4827 | Dir=REVERSE
✅ V1ExtraB: 133.7V | 5.848A | 633.7W | PF=0.80 | F=58.81Hz | E=4403 | Dir=FORWARD
🧩 Archivo unificado actualizado: Medidor1.csv

🏠 Vivienda2 - Ángel
✅ V2FaseA: 131.4V | 1.102A | 128.1W | PF=0.88 | F=59.13Hz | E=160184
✅ V2FaseB: 134.8V | 0.045A | 4.5W | PF=0.75 | F=59.15Hz | E=23260
🧩 Archivo unificado actualizado: Medidor2.csv

🏠 Vivienda3 - María
✅ V3FaseA: 123.9V | 2.570A | 129.0W | PF=0.40 | F=59.13Hz | E=38253
✅ V3Fas